## 1. Check Kaggle Credentials

This cell checks whether the Kaggle username and API key are available from Colab Secrets. These credentials are needed before downloading the competition data.


In [ ]:
from google.colab import userdata

username = userdata.get("KAGGLE_USERNAME")
key = userdata.get("KAGGLE_KEY")

print("Username exists:", username is not None)
print("Key exists:", key is not None)

## 2. Download Competition Data

This cell loads Kaggle credentials into environment variables, downloads the competition files, and prints only a short folder summary. The full file structure is inspected in the next section.


In [ ]:
from google.colab import userdata
import os

import kagglehub

# Get Kaggle credentials from Colab Secrets.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Check that credentials were loaded.
print("Username loaded:", bool(os.environ.get("KAGGLE_USERNAME")))
print("Key loaded:", bool(os.environ.get("KAGGLE_KEY")))

# Download directly into /content so it is visible in Colab Files.
path = kagglehub.competition_download(
    "predict-ai-model-runtime",
    output_dir="/content/predict-ai-model-runtime"
)

print("Downloaded to:", path)

# Keep this output short. Detailed file counts are shown in the inspection section.
print("\nTop-level files/folders:")
for item in sorted(os.listdir(path)):
    print("-", item)


## 3. Inspect the Kaggle Dataset

This section checks the downloaded folder structure, sample submission format, and the keys/shapes inside representative `.npz` graph files.

The folder names combine different ideas:

- `layout` / `tile`: the compiler optimization task.
- `xla` / `nlp`: the graph family. `xla` is general XLA computation graphs, while `nlp` is NLP/BERT-style graphs.
- `default` / `random`: the source of layout configurations. This only exists for `layout`.
- `train` / `valid` / `test`: the normal ML split. Train and valid have runtimes; test is for final submission.

A practical first approach is to treat the competition as five separate datasets/models:

```text
tile:xla
layout:xla:default
layout:xla:random
layout:nlp:default
layout:nlp:random
```

Each `.npz` file is one graph with many candidate configurations. For training, create one row per configuration, learn to predict runtime, then rank the test configurations by predicted runtime.

```text
For each collection:
  train/*.npz -> train one model
  valid/*.npz -> evaluate ranking quality
  test/*.npz  -> predict config ranking for submission.csv
```

### 3.1 Locate the Data Folder

This cell finds the Kaggle download folder and defines the five collection paths used throughout the notebook.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

def resolve_data_root():
    candidates = []
    if "path" in globals():
        candidates.append(Path(path))
    candidates.extend([
        Path("/content/predict-ai-model-runtime"),
        Path.cwd() / "predict-ai-model-runtime",
        Path.cwd(),
    ])

    for candidate in candidates:
        if (candidate / "npz_all" / "npz").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the dataset root. Run the Kaggle download cell first, "
        "or update resolve_data_root() with the correct folder."
    )

DATA_ROOT = resolve_data_root()
NPZ_ROOT = DATA_ROOT / "npz_all" / "npz"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"

collections = {
    "tile:xla": NPZ_ROOT / "tile" / "xla",
    "layout:xla:default": NPZ_ROOT / "layout" / "xla" / "default",
    "layout:xla:random": NPZ_ROOT / "layout" / "xla" / "random",
    "layout:nlp:default": NPZ_ROOT / "layout" / "nlp" / "default",
    "layout:nlp:random": NPZ_ROOT / "layout" / "nlp" / "random",
}

print("DATA_ROOT:", DATA_ROOT)
print("NPZ_ROOT:", NPZ_ROOT)
print("sample_submission exists:", SAMPLE_SUBMISSION_PATH.exists())


### 3.2 Count Files by Collection and Split

This table shows how many `.npz` graph files exist in each `train`, `valid`, and `test` folder.


In [ ]:
def split_files(collection, split):
    return sorted((collections[collection] / split).glob("*.npz"))

rows = []
for collection in collections:
    for split in ["train", "valid", "test"]:
        files = split_files(collection, split)
        rows.append({
            "collection": collection,
            "split": split,
            "num_files": len(files),
            "example_file": files[0].name if files else None,
        })

counts_df = pd.DataFrame(rows)
display(counts_df)


### 3.3 Inspect the Sample Submission

The final output must follow this format: one row per test `.npz` file, with `TopConfigs` as semicolon-separated configuration indices.


In [ ]:
if SAMPLE_SUBMISSION_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print("sample_submission.csv shape:", sample_submission.shape)
    display(sample_submission.head())
else:
    print("sample_submission.csv not found at", SAMPLE_SUBMISSION_PATH)


### 3.4 Inspect Representative `.npz` Files

This cell opens one representative file per collection and prints the arrays inside it. The key things to notice are:

- `node_feat`, `node_opcode`, and `edge_index` describe the graph.
- `config_feat` appears in `tile:xla`.
- `node_config_feat` appears in `layout:*`.
- `config_runtime` is the target label in `train` and `valid`.


In [ ]:
def summarize_array(name, array):
    summary = {
        "key": name,
        "shape": array.shape,
        "dtype": str(array.dtype),
    }
    if np.issubdtype(array.dtype, np.number) and array.size > 0:
        finite_values = array[np.isfinite(array)]
        if finite_values.size > 0:
            summary.update({
                "min": float(np.min(finite_values)),
                "max": float(np.max(finite_values)),
                "mean": float(np.mean(finite_values)),
            })
    return summary

def inspect_npz_file(npz_path, max_preview_keys=20):
    print("\nFile:", npz_path)
    with np.load(npz_path) as data:
        keys = list(data.keys())
        print("keys:", keys)
        summaries = [summarize_array(key, data[key]) for key in keys[:max_preview_keys]]
    display(pd.DataFrame(summaries))

for collection in collections:
    files = split_files(collection, "train")
    if files:
        print("\nCollection:", collection, "| split: train")
        inspect_npz_file(files[0])
    else:
        print("\nNo train files found for", collection)


### 3.5 Preview a Small Slice of the Raw Data

The previous table is only a summary. This cell shows a small slice of the actual raw arrays inside one `.npz` file.

We only print a few rows/columns because the full arrays can contain thousands of numbers.


In [ ]:
RAW_PREVIEW_COLLECTION = "tile:xla"
RAW_PREVIEW_SPLIT = "train"
RAW_PREVIEW_FILE_INDEX = 0

raw_preview_file = split_files(RAW_PREVIEW_COLLECTION, RAW_PREVIEW_SPLIT)[RAW_PREVIEW_FILE_INDEX]
print("Previewing raw data from:", raw_preview_file)

with np.load(raw_preview_file) as data:
    print("\nnode_feat: first 2 nodes, first 10 features")
    display(pd.DataFrame(data["node_feat"][:2, :10]))

    print("\nnode_opcode: operation type ID for each node")
    display(pd.DataFrame({
        "node_id": np.arange(len(data["node_opcode"])),
        "node_opcode": data["node_opcode"],
    }).head(20))

    print("\nedge_index: first 10 directed edges")
    display(pd.DataFrame(data["edge_index"][:10], columns=["source_node", "target_node"]))

    if "config_feat" in data:
        print("\nconfig_feat: first 3 configs, first 10 features")
        display(pd.DataFrame(data["config_feat"][:3, :10]))

    if "node_config_feat" in data:
        print("\nnode_config_feat: config 0, first 3 configurable nodes, first 10 features")
        display(pd.DataFrame(data["node_config_feat"][0, :3, :10]))

    if "config_runtime" in data:
        print("\nconfig_runtime: first 10 runtimes")
        display(pd.DataFrame({
            "config_index": np.arange(min(10, len(data["config_runtime"]))),
            "runtime": data["config_runtime"][:10],
        }))


## 4. Data Checks Instead of Traditional Cleaning

For this competition, "cleaning" is mostly structural checking rather than normal CSV cleaning.

We want to confirm:

- each file has one runtime per configuration in `train` and `valid`
- `test` runtimes are dummy zeros and should not be used as labels
- `tile:xla` uses `config_feat`
- `layout:*` uses `node_config_feat`
- padded values such as `-1` exist in layout configuration features


In [ ]:
def get_num_configs(data):
    if "config_feat" in data:
        return data["config_feat"].shape[0]
    if "node_config_feat" in data:
        return data["node_config_feat"].shape[0]
    raise KeyError("Could not find config_feat or node_config_feat")

check_rows = []

# We inspect only the first few files per split so this cell stays fast.
for collection, collection_dir in collections.items():
    for split in ["train", "valid", "test"]:
        files = sorted((collection_dir / split).glob("*.npz"))[:5]
        for file_path in files:
            with np.load(file_path) as data:
                n_configs = get_num_configs(data)
                runtimes = data["config_runtime"] if "config_runtime" in data else None
                has_tile_config = "config_feat" in data
                has_layout_config = "node_config_feat" in data

                layout_padding_fraction = None
                if has_layout_config:
                    layout_config = data["node_config_feat"]
                    layout_padding_fraction = float(np.mean(layout_config == -1))

                check_rows.append({
                    "collection": collection,
                    "split": split,
                    "file": file_path.name,
                    "n_configs": n_configs,
                    "has_tile_config_feat": has_tile_config,
                    "has_layout_node_config_feat": has_layout_config,
                    "layout_padding_fraction": layout_padding_fraction,
                    "runtime_shape": None if runtimes is None else runtimes.shape,
                    "runtime_min": None if runtimes is None else int(np.min(runtimes)),
                    "runtime_max": None if runtimes is None else int(np.max(runtimes)),
                    "runtime_matches_configs": None if runtimes is None else len(runtimes) == n_configs,
                })

checks_df = pd.DataFrame(check_rows)
display(checks_df)

print("Reminder: test config_runtime values are expected to be dummy zeros, not training labels.")


## 5. Visualize One Sample Graph

Each `.npz` file stores a computation graph:

- `node_feat`: numeric features for each operation node
- `node_opcode`: operation type ID for each node
- `edge_index`: directed edges between nodes

In the plot, the numbers written on the circles are just node IDs, such as node `1`, node `8`, or node `14`. They are not runtimes, ranks, or configuration numbers.

`node_opcode` is the actual operation-type value stored in the data. The node color is just a visual way to display `node_opcode`, so nodes with different operation types appear in different colors. The colors do not mean fast or slow.

Some graphs have thousands of nodes, so this visualization samples a smaller subgraph. It is only for understanding the data; model training does not depend on it.


In [ ]:
import importlib.util
import subprocess
import sys

for package_name, import_name in [("networkx", "networkx"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import matplotlib.pyplot as plt
import networkx as nx

def load_graph_arrays(npz_path):
    with np.load(npz_path) as data:
        return {
            "node_feat": data["node_feat"],
            "node_opcode": data["node_opcode"],
            "edge_index": data["edge_index"],
        }

def sample_nodes_for_visualization(edge_index, num_nodes, max_nodes=80, seed=42):
    if num_nodes <= max_nodes:
        return np.arange(num_nodes)

    # Start around a high-degree node so the sample is likely to be connected.
    degree = np.bincount(edge_index.reshape(-1), minlength=num_nodes)
    start_node = int(np.argmax(degree))

    selected = {start_node}
    frontier = {start_node}

    # Expand from the start node. For visualization, we treat edges as links both ways.
    while frontier and len(selected) < max_nodes:
        next_frontier = set()
        for src, dst in edge_index:
            src = int(src)
            dst = int(dst)
            if src in frontier and dst not in selected:
                next_frontier.add(dst)
            if dst in frontier and src not in selected:
                next_frontier.add(src)

        selected.update(list(next_frontier)[: max_nodes - len(selected)])
        frontier = next_frontier

    # If the connected region is small, fill the rest randomly.
    if len(selected) < max_nodes:
        rng = np.random.default_rng(seed)
        remaining = np.setdiff1d(np.arange(num_nodes), np.array(sorted(selected)))
        fill = rng.choice(remaining, size=min(max_nodes - len(selected), len(remaining)), replace=False)
        selected.update(fill.tolist())

    return np.array(sorted(selected), dtype=int)

def visualize_npz_graph(npz_path, max_nodes=80, title=None):
    arrays = load_graph_arrays(npz_path)
    node_opcode = arrays["node_opcode"]
    edge_index = arrays["edge_index"]
    num_nodes = len(node_opcode)

    selected_nodes = sample_nodes_for_visualization(edge_index, num_nodes, max_nodes=max_nodes)
    selected_set = set(selected_nodes.tolist())

    graph = nx.DiGraph()
    for node_id in selected_nodes:
        graph.add_node(int(node_id), opcode=int(node_opcode[node_id]))

    for src, dst in edge_index:
        src = int(src)
        dst = int(dst)
        if src in selected_set and dst in selected_set:
            graph.add_edge(src, dst)

    plt.figure(figsize=(13, 9))
    pos = nx.spring_layout(graph, seed=42, k=0.7)
    colors = [graph.nodes[n]["opcode"] for n in graph.nodes]

    nodes = nx.draw_networkx_nodes(
        graph,
        pos,
        node_color=colors,
        cmap="tab20",
        node_size=180,
        alpha=0.95,
    )
    nx.draw_networkx_edges(graph, pos, arrows=True, arrowsize=8, width=0.7, alpha=0.35)
    nx.draw_networkx_labels(graph, pos, labels={n: str(n) for n in graph.nodes}, font_size=7)
    plt.colorbar(nodes, label="node_opcode")
    plt.title(title or f"{npz_path.name}: {graph.number_of_nodes()} sampled nodes, {graph.number_of_edges()} sampled edges")
    plt.axis("off")
    plt.show()

    print("Original graph nodes:", num_nodes)
    print("Original graph edges:", len(edge_index))
    print("Displayed nodes:", graph.number_of_nodes())
    print("Displayed edges:", graph.number_of_edges())
    print("Unique opcodes in displayed subgraph:", sorted(set(colors))[:30])

# Change these values if you want to inspect a different collection or split.
VIS_COLLECTION = "tile:xla"
VIS_SPLIT = "train"
VIS_FILE_INDEX = 0
MAX_NODES_TO_DRAW = 80

visual_files = sorted((collections[VIS_COLLECTION] / VIS_SPLIT).glob("*.npz"))
sample_graph_path = visual_files[VIS_FILE_INDEX]

print("Visualizing:", sample_graph_path)
visualize_npz_graph(sample_graph_path, max_nodes=MAX_NODES_TO_DRAW, title=f"Sample graph: {VIS_COLLECTION} / {VIS_SPLIT}")
